In [1]:
# ============================================================
# SMART TEXT SUMMARIZATION & ANALYSIS SYSTEM
# SIMPLE WORKING VERSION FOR GOOGLE COLAB
# ============================================================

# Install libraries
!pip -q install gradio nltk textstat


# ============================================================
# IMPORT LIBRARIES
# ============================================================

import re
import nltk
import textstat
import gradio as gr

from collections import Counter

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer


# ============================================================
# DOWNLOAD NLTK DATA
# ============================================================

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("vader_lexicon", quiet=True)


# ============================================================
# INITIALIZE
# ============================================================

stop_words = set(stopwords.words("english"))

sentiment_analyzer = SentimentIntensityAnalyzer()


# ============================================================
# TEXT CLEANING
# ============================================================

def clean_text(text):

    if not text:
        return ""

    text = re.sub(r"\s+", " ", text)

    return text.strip()


# ============================================================
# SUMMARY
# ============================================================

def generate_summary(text):

    sentences = sent_tokenize(text)

    if len(sentences) <= 3:
        return text

    words = word_tokenize(text.lower())

    # Remove stopwords and punctuation
    useful_words = [
        word
        for word in words
        if word.isalpha() and word not in stop_words
    ]

    # Word frequency
    frequency = Counter(useful_words)

    # Score sentences
    sentence_scores = {}

    for sentence in sentences:

        sentence_words = word_tokenize(
            sentence.lower()
        )

        score = 0

        for word in sentence_words:

            if word in frequency:
                score += frequency[word]

        sentence_scores[sentence] = score

    # Select top 3 sentences
    top_sentences = sorted(
        sentence_scores,
        key=sentence_scores.get,
        reverse=True
    )[:3]

    # Maintain original order
    summary = []

    for sentence in sentences:

        if sentence in top_sentences:
            summary.append(sentence)

    return " ".join(summary)


# ============================================================
# KEYWORD EXTRACTION
# ============================================================

def get_keywords(text):

    words = word_tokenize(text.lower())

    useful_words = [
        word
        for word in words
        if word.isalpha() and word not in stop_words
    ]

    frequency = Counter(useful_words)

    top_words = frequency.most_common(10)

    if not top_words:
        return "No keywords found."

    result = ""

    for word, count in top_words:

        result += f"• {word} — {count} time(s)\n"

    return result


# ============================================================
# TEXT STATISTICS
# ============================================================

def get_statistics(text):

    sentences = sent_tokenize(text)

    words = word_tokenize(text)

    actual_words = [
        word
        for word in words
        if word.isalpha()
    ]

    character_count = len(text)

    word_count = len(actual_words)

    sentence_count = len(sentences)

    paragraph_count = len(
        [
            p for p in text.split("\n")
            if p.strip()
        ]
    )

    if word_count > 0:

        average_word_length = (
            sum(len(word) for word in actual_words)
            / word_count
        )

    else:

        average_word_length = 0

    if sentence_count > 0:

        average_sentence_length = (
            word_count / sentence_count
        )

    else:

        average_sentence_length = 0

    return f"""
📌 Characters: {character_count}

📌 Words: {word_count}

📌 Sentences: {sentence_count}

📌 Paragraphs: {paragraph_count}

📌 Average Word Length: {average_word_length:.2f}

📌 Average Sentence Length: {average_sentence_length:.2f}
"""


# ============================================================
# SENTIMENT ANALYSIS
# ============================================================

def get_sentiment(text):

    scores = sentiment_analyzer.polarity_scores(text)

    compound = scores["compound"]

    if compound >= 0.05:

        sentiment = "Positive 😊"

    elif compound <= -0.05:

        sentiment = "Negative 😞"

    else:

        sentiment = "Neutral 😐"

    return f"""
Overall Sentiment: {sentiment}

Positive: {scores['pos']:.3f}

Negative: {scores['neg']:.3f}

Neutral: {scores['neu']:.3f}

Compound: {scores['compound']:.3f}
"""


# ============================================================
# READABILITY
# ============================================================

def get_readability(text):

    try:

        reading_score = textstat.flesch_reading_ease(text)

        grade = textstat.flesch_kincaid_grade(text)

        if reading_score >= 90:
            level = "Very Easy"

        elif reading_score >= 80:
            level = "Easy"

        elif reading_score >= 70:
            level = "Fairly Easy"

        elif reading_score >= 60:
            level = "Standard"

        elif reading_score >= 50:
            level = "Fairly Difficult"

        elif reading_score >= 30:
            level = "Difficult"

        else:
            level = "Very Difficult"

        return f"""
📖 Reading Ease Score: {reading_score:.2f}

🎓 Estimated Grade Level: {grade:.2f}

📚 Reading Difficulty: {level}
"""

    except Exception:

        return "Readability calculation unavailable."


# ============================================================
# MAIN FUNCTION
# ============================================================

def analyze_text(text):

    if not text or not text.strip():

        return (
            "⚠️ Please enter some text.",
            "",
            "",
            "",
            ""
        )

    # Clean text
    text = clean_text(text)

    # Generate outputs
    summary = generate_summary(text)

    keywords = get_keywords(text)

    statistics = get_statistics(text)

    sentiment = get_sentiment(text)

    readability = get_readability(text)

    return (
        summary,
        keywords,
        statistics,
        sentiment,
        readability
    )


# ============================================================
# SAMPLE TEXT
# ============================================================

sample_text = """
Artificial intelligence is a rapidly growing field of computer
science. It enables machines to perform tasks that normally require
human intelligence. These tasks include learning, reasoning,
problem solving, understanding language and recognizing images.

Artificial intelligence is used in many different industries.
Healthcare uses AI to analyze medical images and support doctors.
Education uses intelligent systems to provide personalized learning
experiences. Banks use AI to detect suspicious transactions and
improve customer service.

Machine learning is an important part of artificial intelligence.
Machine learning allows computers to learn patterns from data without
being explicitly programmed for every possible situation.

Deep learning is another important technology. It uses artificial
neural networks with multiple layers to process large amounts of
data. Deep learning is widely used in image recognition, speech
recognition and natural language processing.

Although artificial intelligence provides many benefits, it is also
important to consider privacy, security and ethical issues. Responsible
development of AI can help society use the technology safely and
effectively.
"""


# ============================================================
# GRADIO UI
# ============================================================

with gr.Blocks(
    title="Smart Text Summarization & Analysis System"
) as app:

    gr.Markdown(
        """
        # 🧠 Smart Text Summarization & Analysis System

        ### Natural Language Processing Based Text Analysis

        **Summary • Keywords • Statistics • Sentiment • Readability**
        """
    )

    # INPUT
    text_input = gr.Textbox(
        label="📝 Enter Your Text",
        value=sample_text,
        placeholder="Paste your text here...",
        lines=15
    )

    # BUTTON
    with gr.Row():

        analyze_button = gr.Button(
            "🔍 Analyze Text",
            variant="primary"
        )

        clear_button = gr.Button(
            "🗑️ Clear"
        )

    # RESULTS
    gr.Markdown("## 📊 Analysis Results")

    summary_output = gr.Textbox(
        label="📄 Summary",
        lines=6
    )

    keywords_output = gr.Textbox(
        label="🔑 Important Keywords",
        lines=8
    )

    statistics_output = gr.Textbox(
        label="📈 Text Statistics",
        lines=8
    )

    sentiment_output = gr.Textbox(
        label="😊 Sentiment Analysis",
        lines=8
    )

    readability_output = gr.Textbox(
        label="📚 Readability Analysis",
        lines=7
    )


    # ANALYZE BUTTON
    analyze_button.click(
        fn=analyze_text,
        inputs=text_input,
        outputs=[
            summary_output,
            keywords_output,
            statistics_output,
            sentiment_output,
            readability_output
        ]
    )


    # CLEAR BUTTON
    clear_button.click(
        fn=lambda: (
            "",
            "",
            "",
            "",
            ""
        ),
        inputs=[],
        outputs=[
            text_input,
            summary_output,
            keywords_output,
            statistics_output,
            sentiment_output
        ]
    )


# ============================================================
# LAUNCH
# ============================================================

print("🚀 Starting application...")

app.launch(
    share=True
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 12.6 MB/s eta 0:00:00
🚀 Starting application...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://33347d833274c7f21e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [2]:
# ============================================================
# SMART TEXT SUMMARIZATION & ANALYSIS SYSTEM
# STEP 3 - COMPLETE CODE
# Text Input + TXT File Upload
# ============================================================

# =========================
# 1. INSTALL LIBRARIES
# =========================

!pip -q install gradio nltk textstat


# =========================
# 2. IMPORT LIBRARIES
# =========================

import re
import nltk
import textstat
import gradio as gr

from collections import Counter

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer


# =========================
# 3. DOWNLOAD NLTK DATA
# =========================

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("vader_lexicon", quiet=True)


# =========================
# 4. INITIALIZE
# =========================

stop_words = set(stopwords.words("english"))

sentiment_analyzer = SentimentIntensityAnalyzer()


# ============================================================
# 5. CLEAN TEXT
# ============================================================

def clean_text(text):

    if not text:
        return ""

    text = re.sub(r"\s+", " ", text)

    return text.strip()


# ============================================================
# 6. READ TXT FILE
# ============================================================

def read_file(file_path):

    if file_path is None:
        return ""

    try:

        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as file:

            return file.read()

    except Exception as e:

        return f"Error reading file: {e}"


# ============================================================
# 7. SUMMARY
# ============================================================

def generate_summary(text):

    sentences = sent_tokenize(text)

    if len(sentences) <= 3:
        return text

    words = word_tokenize(text.lower())

    useful_words = [
        word
        for word in words
        if word.isalpha()
        and word not in stop_words
    ]

    frequency = Counter(useful_words)

    if not frequency:
        return text

    sentence_scores = {}

    for sentence in sentences:

        sentence_words = word_tokenize(
            sentence.lower()
        )

        score = 0

        for word in sentence_words:

            if word in frequency:
                score += frequency[word]

        sentence_scores[sentence] = score

    top_sentences = sorted(
        sentence_scores,
        key=sentence_scores.get,
        reverse=True
    )[:3]

    summary = []

    for sentence in sentences:

        if sentence in top_sentences:
            summary.append(sentence)

    return " ".join(summary)


# ============================================================
# 8. KEYWORD EXTRACTION
# ============================================================

def get_keywords(text):

    words = word_tokenize(text.lower())

    useful_words = [
        word
        for word in words
        if word.isalpha()
        and word not in stop_words
    ]

    frequency = Counter(useful_words)

    top_words = frequency.most_common(10)

    if not top_words:
        return "No keywords found."

    result = ""

    for word, count in top_words:

        result += (
            f"• {word} — {count} time(s)\n"
        )

    return result


# ============================================================
# 9. TEXT STATISTICS
# ============================================================

def get_statistics(text):

    sentences = sent_tokenize(text)

    words = word_tokenize(text)

    actual_words = [
        word
        for word in words
        if word.isalpha()
    ]

    character_count = len(text)

    word_count = len(actual_words)

    sentence_count = len(sentences)

    paragraph_count = len(
        [
            p
            for p in text.split("\n")
            if p.strip()
        ]
    )

    if word_count > 0:

        average_word_length = (
            sum(
                len(word)
                for word in actual_words
            )
            / word_count
        )

    else:

        average_word_length = 0

    if sentence_count > 0:

        average_sentence_length = (
            word_count / sentence_count
        )

    else:

        average_sentence_length = 0

    return f"""
📌 Characters: {character_count}

📌 Words: {word_count}

📌 Sentences: {sentence_count}

📌 Paragraphs: {paragraph_count}

📌 Average Word Length: {average_word_length:.2f}

📌 Average Sentence Length: {average_sentence_length:.2f}
"""


# ============================================================
# 10. SENTIMENT ANALYSIS
# ============================================================

def get_sentiment(text):

    scores = sentiment_analyzer.polarity_scores(text)

    compound = scores["compound"]

    if compound >= 0.05:

        sentiment = "Positive 😊"

    elif compound <= -0.05:

        sentiment = "Negative 😞"

    else:

        sentiment = "Neutral 😐"

    return f"""
Overall Sentiment: {sentiment}

Positive: {scores['pos']:.3f}

Negative: {scores['neg']:.3f}

Neutral: {scores['neu']:.3f}

Compound: {scores['compound']:.3f}
"""


# ============================================================
# 11. READABILITY ANALYSIS
# ============================================================

def get_readability(text):

    try:

        reading_score = (
            textstat.flesch_reading_ease(text)
        )

        grade = (
            textstat.flesch_kincaid_grade(text)
        )

        if reading_score >= 90:

            level = "Very Easy"

        elif reading_score >= 80:

            level = "Easy"

        elif reading_score >= 70:

            level = "Fairly Easy"

        elif reading_score >= 60:

            level = "Standard"

        elif reading_score >= 50:

            level = "Fairly Difficult"

        elif reading_score >= 30:

            level = "Difficult"

        else:

            level = "Very Difficult"

        return f"""
📖 Reading Ease Score: {reading_score:.2f}

🎓 Estimated Grade Level: {grade:.2f}

📚 Reading Difficulty: {level}
"""

    except Exception:

        return "Readability calculation unavailable."


# ============================================================
# 12. MAIN ANALYSIS FUNCTION
# ============================================================

def analyze_text(text):

    if not text or not text.strip():

        return (
            "⚠️ Please enter or upload some text.",
            "",
            "",
            "",
            ""
        )

    text = clean_text(text)

    summary = generate_summary(text)

    keywords = get_keywords(text)

    statistics = get_statistics(text)

    sentiment = get_sentiment(text)

    readability = get_readability(text)

    return (
        summary,
        keywords,
        statistics,
        sentiment,
        readability
    )


# ============================================================
# 13. SAMPLE TEXT
# ============================================================

sample_text = """
Artificial intelligence is a rapidly growing field of computer
science. It enables machines to perform tasks that normally require
human intelligence. These tasks include learning, reasoning,
problem solving, understanding language and recognizing images.

Artificial intelligence is used in many different industries.
Healthcare uses AI to analyze medical images and support doctors.
Education uses intelligent systems to provide personalized learning
experiences. Banks use AI to detect suspicious transactions and
improve customer service.

Machine learning is an important part of artificial intelligence.
Machine learning allows computers to learn patterns from data without
being explicitly programmed for every possible situation.

Deep learning is another important technology. It uses artificial
neural networks with multiple layers to process large amounts of
data. Deep learning is widely used in image recognition, speech
recognition and natural language processing.

Although artificial intelligence provides many benefits, it is also
important to consider privacy, security and ethical issues.
Responsible development of AI can help society use the technology
safely and effectively.
"""


# ============================================================
# 14. CREATE GRADIO APP
# ============================================================

with gr.Blocks(
    title="Smart Text Summarization & Analysis System"
) as app:

    # HEADER
    gr.Markdown(
        """
        # 🧠 Smart Text Summarization & Analysis System

        ### Natural Language Processing Based Text Analysis

        **Summary • Keywords • Statistics • Sentiment • Readability**
        """
    )

    # TEXT INPUT
    text_input = gr.Textbox(
        label="📝 Enter Your Text",
        value=sample_text,
        placeholder="Paste your text here...",
        lines=15
    )

    # FILE UPLOAD
    file_input = gr.File(
        label="📁 Upload TXT File",
        file_types=[".txt"],
        type="filepath"
    )

    # LOAD FILE BUTTON
    load_file_button = gr.Button(
        "📂 Load TXT File"
    )

    # BUTTONS
    with gr.Row():

        analyze_button = gr.Button(
            "🔍 Analyze Text",
            variant="primary"
        )

        clear_button = gr.Button(
            "🗑️ Clear"
        )

    # RESULTS
    gr.Markdown(
        "## 📊 Analysis Results"
    )

    summary_output = gr.Textbox(
        label="📄 Summary",
        lines=6
    )

    keywords_output = gr.Textbox(
        label="🔑 Important Keywords",
        lines=8
    )

    statistics_output = gr.Textbox(
        label="📈 Text Statistics",
        lines=8
    )

    sentiment_output = gr.Textbox(
        label="😊 Sentiment Analysis",
        lines=8
    )

    readability_output = gr.Textbox(
        label="📚 Readability Analysis",
        lines=7
    )

    # ========================================================
    # LOAD FILE
    # ========================================================

    load_file_button.click(
        fn=read_file,
        inputs=file_input,
        outputs=text_input
    )

    # ========================================================
    # ANALYZE
    # ========================================================

    analyze_button.click(
        fn=analyze_text,
        inputs=text_input,
        outputs=[
            summary_output,
            keywords_output,
            statistics_output,
            sentiment_output,
            readability_output
        ]
    )

    # ========================================================
    # CLEAR
    # ========================================================

    clear_button.click(
        fn=lambda: (
            "",
            None,
            "",
            "",
            "",
            "",
            ""
        ),
        inputs=[],
        outputs=[
            text_input,
            file_input,
            summary_output,
            keywords_output,
            statistics_output,
            sentiment_output,
            readability_output
        ]
    )

    # FOOTER
    gr.Markdown(
        """
        ---
        ### 🔬 Technologies Used

        **Python | NLTK | TextStat | VADER | Gradio**

        **Smart Text Summarization & Analysis System**
        """
    )


# ============================================================
# 15. LAUNCH
# ============================================================

print("🚀 Starting Smart Text Summarization & Analysis System...")

app.launch(
    share=True
)

🚀 Starting Smart Text Summarization & Analysis System...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1e9e5b002c97e9eda6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
